<a href="https://colab.research.google.com/github/427paul/AI_Agent/blob/main/%5BBDA%5D_12%EC%A3%BC%EC%B0%A8_RAG_%ED%8C%8C%EC%9D%B4%ED%94%84%EB%9D%BC%EC%9D%B8_%EC%8B%A4%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧩 12주차 · RAG 파이프라인 실습

## 오늘 만들 것
> **위니브마켓 반품 정책 챗봇** — 우리 문서를 읽고 답하는 AI 🤖

11주차에서 만든 **벡터 DB(저장소)** 를 LLM에 연결해,
"검색한 문서로 답하는" **RAG 파이프라인**을 한 단계씩 완성합니다.

### 🎯 학습목표
1. 문서를 청크로 나누고(분할) 벡터 DB에 저장한다
2. 리트리버로 관련 문서를 검색한다
3. 검색 결과를 프롬프트에 넣어(증강) LLM이 답하게 한다
4. LCEL(`|`)로 5단계를 하나의 체인으로 연결한다

### 📋 RAG 5단계 & 셀 번호
| 단계 | 이름 | 셀 |
| --- | --- | --- |
| ① | 로드 & 분할 | 1️⃣ 2️⃣ 3️⃣ |
| ② | 임베딩 & 저장 | 4️⃣ |
| ③ | 검색 (Retriever) | 5️⃣ |
| ④ | 증강 (프롬프트) | 6️⃣ 7️⃣ |
| ⑤ | 생성 (LLM) | 8️⃣ |
| — | 연결 & 비교 | 9️⃣ 🔟 |

> 💡 **위에서부터 순서대로** 셀을 실행하세요. (단축키: `Shift + Enter`)
> Google Colab 기준으로 작성되었습니다.

---
## 0. 준비 — 라이브러리 설치

먼저 필요한 라이브러리를 설치합니다. (30초 정도 걸려요 ☕)

In [1]:
!pip install -q langchain-google-genai langchain-chroma chromadb langchain-text-splitters

print("✅ 설치 완료!")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.3.30 requires langchain-core<1.0.0,>=0.3.85, but you have langchain-core 1.4.9 which is incompatible.
langchain-classic 1.0.8 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.3.11 which is incompatible.
langchain-huggingface 0.3.1 requires langchain-core<1.0.0,>=0.3.70, but you have langchain-core 1.4.9 which is incompatible.
✅ 설치 완료!


In [2]:
!pip install langchain langchain-community langchain-openai chromadb tiktoken -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-google-genai 4.2.7 requires langchain-core<2.0.0,>=1.4.7, but you have langchain-core 0.3.86 which is incompatible.
langgraph-sdk 0.4.2 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.3.86 which is incompatible.
langgraph 1.2.9 requires langchain-core<2,>=1.4.7, but you have langchain-core 0.3.86 which is incompatible.
langchain-classic 1.0.8 requires langchain-core<2.0.0,>=1.4.4, but you have langchain-core 0.3.86 which is incompatible.
langchain-classic 1.0.8 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.3.11 which is incompatible.
langchain-chroma 1.1.0 requires langchain-core<2.0.0,>=1.1.3, but you have langchain-core 0.3.86 which is incompatible.
langgraph-prebuilt 1.1.0 requires langchain-core>=1.3.1, but you have langchain-core 0.3.86 whi

In [3]:
!pip install -U langchain-core  google-genai

  Using cached langchain-1.3.14-py3-none-any.whl.metadata (6.1 kB)
  Using cached langchain_core-1.4.9-py3-none-any.whl.metadata (4.7 kB)
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 1.8 MB/s eta 0:00:00
  Using cached langchain_text_splitters-1.1.2-py3-none-any.whl.metadata (3.3 kB)
Using cached langchain-1.3.14-py3-none-any.whl (139 kB)
Using cached langchain_core-1.4.9-py3-none-any.whl (558 kB)
Using cached langchain_community-0.4.2-py3-none-any.whl (2.4 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.8 MB/s eta 0:00:00
Using cached langchain_text_splitters-1.1.2-py3-none-any.whl (35 kB)
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.86
    Uninstalling langchain-core-0.3.86:
      Successfully uninstalled langchain-core-0.3.86
  Attempting uninstall: langchain-text-splitters
    Found existing installation: langchain-text-splitters 0.3.

## 0-1. Google API 키 설정 🔑

Gemini 모델을 쓰려면 **Google API 키**가 필요합니다.
- 발급(무료): https://aistudio.google.com/apikey
- 아래 셀을 실행하면 입력창이 뜹니다. 키를 붙여넣고 **Enter**.

In [24]:
import os
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = getpass("Google API 키를 입력하세요: ")
print("✅ API 키 설정 완료!")

Google API 키를 입력하세요: ··········
✅ API 키 설정 완료!


In [33]:
os.environ["HF_TOKEN"] = getpass("HuggingFace Token을 입력하세요: ")
print("✅ Token 설정 완료!")

HuggingFace Token을 입력하세요: ··········
✅ Token 설정 완료!


---
## 1️⃣ 문서 준비

RAG의 재료가 될 **원본 문서**를 준비합니다.
여기서는 예시로 '위니브마켓 반품 정책'을 사용합니다.

> 💡 실제로는 PDF·txt 파일을 불러오지만, 오늘은 개념에 집중하기 위해 문자열로 준비합니다.

In [5]:
# 1️⃣ 원본 문서 (위니브마켓 반품·교환 안내)
policy_text = """위니브마켓 반품·교환 안내

[반품 신청 기간]
위니브마켓에서 구매하신 상품은 수령일로부터 14일 이내에 반품을 신청할 수 있습니다.
반품 신청은 마이페이지 또는 고객센터(1234-5678)를 통해 접수할 수 있습니다.

[반품 배송비]
단순 변심에 의한 반품의 경우, 왕복 배송비는 고객님께서 부담하셔야 합니다.
다만 상품에 하자가 있거나 오배송된 경우에는 배송비 전액을 위니브마켓이 부담합니다.

[반품이 불가한 상품]
신선식품 및 냉장·냉동 식품은 상품 특성상 반품이 불가합니다.
또한 고객의 사용으로 상품 가치가 현저히 감소한 경우에도 반품이 제한될 수 있습니다.

[교환 안내]
교환은 동일 상품의 색상 또는 사이즈 변경에 한해 1회 무료로 제공됩니다.
다른 상품으로의 교환은 반품 후 재주문으로 처리됩니다.

[환불 시점]
반품 상품이 물류센터에 도착해 검수가 완료되면,
영업일 기준 3일 이내에 결제하신 수단으로 환불이 진행됩니다."""

print(f"📄 문서 길이: {len(policy_text)}자")
print("-" * 40)
print(policy_text)

📄 문서 길이: 472자
----------------------------------------
위니브마켓 반품·교환 안내

[반품 신청 기간]
위니브마켓에서 구매하신 상품은 수령일로부터 14일 이내에 반품을 신청할 수 있습니다.
반품 신청은 마이페이지 또는 고객센터(1234-5678)를 통해 접수할 수 있습니다.

[반품 배송비]
단순 변심에 의한 반품의 경우, 왕복 배송비는 고객님께서 부담하셔야 합니다.
다만 상품에 하자가 있거나 오배송된 경우에는 배송비 전액을 위니브마켓이 부담합니다.

[반품이 불가한 상품]
신선식품 및 냉장·냉동 식품은 상품 특성상 반품이 불가합니다.
또한 고객의 사용으로 상품 가치가 현저히 감소한 경우에도 반품이 제한될 수 있습니다.

[교환 안내]
교환은 동일 상품의 색상 또는 사이즈 변경에 한해 1회 무료로 제공됩니다.
다른 상품으로의 교환은 반품 후 재주문으로 처리됩니다.

[환불 시점]
반품 상품이 물류센터에 도착해 검수가 완료되면,
영업일 기준 3일 이내에 결제하신 수단으로 환불이 진행됩니다.


## 2️⃣ 문서 분할 (Split) ✂️

긴 문서를 **작은 청크**로 나눕니다.
- `chunk_size=500` : 청크 하나의 최대 길이(글자 수)
- `chunk_overlap=50` : 이웃 청크와 겹치는 길이 (문맥이 끊기지 않게)

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 2️⃣ 문서를 청크로 분할
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,     # 청크 최대 길이
    chunk_overlap=50,   # 겹치는 길이
)
chunks = splitter.split_text(policy_text)

print(f"✂️ 총 {len(chunks)}개의 청크로 나뉘었습니다.")

✂️ 총 1개의 청크로 나뉘었습니다.


## 3️⃣ 분할 결과 확인 🔎

나뉜 청크를 하나씩 눈으로 확인해 봅시다.
청크 경계에서 **겹치는 부분(overlap)** 이 어떻게 앞뒤 문맥을 이어주는지 살펴보세요.

In [7]:
# 3️⃣ 청크를 하나씩 출력
for i, chunk in enumerate(chunks):
    print(f"─── 청크 {i}  (길이 {len(chunk)}자) ───")
    print(chunk)
    print()

─── 청크 0  (길이 472자) ───
위니브마켓 반품·교환 안내

[반품 신청 기간]
위니브마켓에서 구매하신 상품은 수령일로부터 14일 이내에 반품을 신청할 수 있습니다.
반품 신청은 마이페이지 또는 고객센터(1234-5678)를 통해 접수할 수 있습니다.

[반품 배송비]
단순 변심에 의한 반품의 경우, 왕복 배송비는 고객님께서 부담하셔야 합니다.
다만 상품에 하자가 있거나 오배송된 경우에는 배송비 전액을 위니브마켓이 부담합니다.

[반품이 불가한 상품]
신선식품 및 냉장·냉동 식품은 상품 특성상 반품이 불가합니다.
또한 고객의 사용으로 상품 가치가 현저히 감소한 경우에도 반품이 제한될 수 있습니다.

[교환 안내]
교환은 동일 상품의 색상 또는 사이즈 변경에 한해 1회 무료로 제공됩니다.
다른 상품으로의 교환은 반품 후 재주문으로 처리됩니다.

[환불 시점]
반품 상품이 물류센터에 도착해 검수가 완료되면,
영업일 기준 3일 이내에 결제하신 수단으로 환불이 진행됩니다.



## 4️⃣ 임베딩 & 벡터 DB 저장 📦

청크를 숫자 벡터로 바꿔(임베딩) **Chroma 벡터 DB**에 저장합니다.

→ 이 부분은 **11주차와 똑같습니다.** (입력이 "문장 3개" → "청크 리스트"로 바뀐 것뿐)

In [13]:
!pip install langchain-chroma

In [9]:
from langchain_community.embeddings import HuggingFaceEmbeddings
# from langchain.vectorstores import Chroma

# Hugging Face 임베딩 모델 (sentence-transformers 기반)
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

/tmp/ipykernel_33295/3788074668.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [15]:
from langchain_chroma import Chroma
vectorstore = Chroma.from_texts(
    texts=chunks,
    embedding=embedding_model
)

print("✅ 벡터 DB 저장 완료!")
print(f"📦 저장된 청크 수: {len(chunks)}개")

✅ 벡터 DB 저장 완료!
📦 저장된 청크 수: 1개


In [12]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

# 4️⃣ 임베딩 모델 준비 + 청크를 벡터 DB에 저장
# embedding = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
embedding = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")
vectorstore = Chroma.from_texts(texts=chunks, embedding=embedding)

print("✅ 벡터 DB 저장 완료!")
print(f"📦 저장된 청크 수: {len(chunks)}개")

ImportError: module ''langchain_core.language_models'.'model_profile'' not found (No module named 'langchain_core.language_models.model_profile')

## 5️⃣ 리트리버(검색기) 만들기 & 검색 테스트 🔍

벡터 DB를 **리트리버**로 바꿉니다. 리트리버는 "질문을 주면 관련 청크를 반환하는" 부품입니다.
`k=3` → 가장 관련 있는 청크 3개를 가져옵니다.

먼저 검색이 잘 되는지 테스트해 봅시다.

In [16]:
# 5️⃣ 벡터 DB → 리트리버로 변환
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 검색 테스트: 질문과 관련된 청크를 찾아본다
test_docs = retriever.invoke("반품 배송비는 누가 내나요?")

print(f"🔍 검색된 청크 {len(test_docs)}개:\n")
for i, doc in enumerate(test_docs):
    print(f"[{i}] {doc.page_content[:50].strip()} ...")

🔍 검색된 청크 1개:

[0] 위니브마켓 반품·교환 안내

[반품 신청 기간]
위니브마켓에서 구매하신 상품은 수령일로부터 ...


## 6️⃣ 프롬프트 만들기 (증강 준비) 📝

검색한 청크를 LLM에게 건네줄 **프롬프트 틀**을 만듭니다.
- `{context}` : 검색된 청크가 들어갈 자리
- `{question}` : 사용자 질문이 들어갈 자리

⭐ **핵심**: "컨텍스트에만 근거해 답하라"는 지시가 **환각을 막아줍니다.**

In [17]:
from langchain_core.prompts import ChatPromptTemplate

# 6️⃣ 프롬프트 틀 만들기
prompt = ChatPromptTemplate.from_template(
    "당신은 위니브마켓 고객센터 상담원입니다.\n"
    "아래 [컨텍스트]에만 근거해 답하세요. "
    "컨텍스트에 없으면 '해당 내용은 안내되어 있지 않습니다'라고 답하세요.\n\n"
    "[컨텍스트]\n{context}\n\n"
    "[질문]\n{question}"
)

print("✅ 프롬프트 준비 완료!\n")
print(prompt.messages[0].prompt.template)

✅ 프롬프트 준비 완료!

당신은 위니브마켓 고객센터 상담원입니다.
아래 [컨텍스트]에만 근거해 답하세요. 컨텍스트에 없으면 '해당 내용은 안내되어 있지 않습니다'라고 답하세요.

[컨텍스트]
{context}

[질문]
{question}


## 7️⃣ 프롬프트에 실제로 채워보기 (증강 확인) 📨

LLM에게 넘어가기 **직전의 프롬프트**가 어떤 모습인지 눈으로 확인해 봅시다.
검색된 청크(`context`)와 질문(`question`)이 빈칸에 채워집니다.

→ 이 완성된 프롬프트가 곧 LLM의 입력이 됩니다.

In [18]:
# 7️⃣ 검색된 청크를 하나의 문자열로 합쳐 프롬프트에 채워보기
question = "반품은 며칠 이내에 가능한가요?"
docs = retriever.invoke(question)
context = "\n\n".join(doc.page_content for doc in docs)

filled = prompt.invoke({"context": context, "question": question})

print("📨 LLM에게 전달될 최종 프롬프트:\n")
print(filled.messages[0].content)

📨 LLM에게 전달될 최종 프롬프트:

당신은 위니브마켓 고객센터 상담원입니다.
아래 [컨텍스트]에만 근거해 답하세요. 컨텍스트에 없으면 '해당 내용은 안내되어 있지 않습니다'라고 답하세요.

[컨텍스트]
위니브마켓 반품·교환 안내

[반품 신청 기간]
위니브마켓에서 구매하신 상품은 수령일로부터 14일 이내에 반품을 신청할 수 있습니다.
반품 신청은 마이페이지 또는 고객센터(1234-5678)를 통해 접수할 수 있습니다.

[반품 배송비]
단순 변심에 의한 반품의 경우, 왕복 배송비는 고객님께서 부담하셔야 합니다.
다만 상품에 하자가 있거나 오배송된 경우에는 배송비 전액을 위니브마켓이 부담합니다.

[반품이 불가한 상품]
신선식품 및 냉장·냉동 식품은 상품 특성상 반품이 불가합니다.
또한 고객의 사용으로 상품 가치가 현저히 감소한 경우에도 반품이 제한될 수 있습니다.

[교환 안내]
교환은 동일 상품의 색상 또는 사이즈 변경에 한해 1회 무료로 제공됩니다.
다른 상품으로의 교환은 반품 후 재주문으로 처리됩니다.

[환불 시점]
반품 상품이 물류센터에 도착해 검수가 완료되면,
영업일 기준 3일 이내에 결제하신 수단으로 환불이 진행됩니다.

[질문]
반품은 며칠 이내에 가능한가요?


## 8️⃣ LLM 준비 (생성) 🧠

답을 만들어줄 **Gemini 모델**을 준비합니다.
- `temperature=0` : 창작이 아니라 사실 응답이므로, 일관된 답을 위해 0으로 설정

> 💡 모델은 `gemini-2.0-flash`를 사용합니다. 필요하면 다른 Gemini 모델로 바꿔도 됩니다.

In [26]:
from langchain_google_genai import ChatGoogleGenerativeAI

# 8️⃣ 답을 생성할 LLM
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)

print("✅ LLM 준비 완료!")

✅ LLM 준비 완료!


In [34]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
# llm = ChatOpenAI(model="gpt-4.1-mini")  # 언어 모델을 gpt-4.1-mini로 설정
# HuggingFace에서 해당 모델을 불러오는 엔드포인트 지정
llm_ep = HuggingFaceEndpoint(repo_id="openai/gpt-oss-20b", task="conversational")

# HuggingFace에서 가져온 모델을 그대로 쓰지 않고,
# LangChain에서 쉽게 쓰도록 감싸는(wrapper) 단계
llm = ChatHuggingFace(llm=llm_ep)

print("✅ LLM 준비 완료!")

✅ LLM 준비 완료!


## 9️⃣ LCEL로 파이프라인 연결 & 실행 🔗

드디어 5단계를 **하나의 체인**으로 잇습니다. `|`(파이프)로 부품을 연결합니다.

```
{context, question}  |  prompt  |  llm  |  StrOutputParser()
   검색 + 질문 전달        조립        생성        문자열로 추출
```

- `format_docs` : 여러 청크를 하나의 문자열로 합치는 함수
- `RunnablePassthrough()` : 질문을 그대로 통과시킴

In [35]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 여러 청크를 하나의 문자열로 합치는 함수
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 9️⃣ LCEL로 RAG 체인 연결
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 실행!
answer = rag_chain.invoke("위니브마켓 반품 정책이 뭐야?")
print("🤖 RAG 답변:\n")
print(answer)

🤖 RAG 답변:

**위니브마켓 반품 정책 요약**

| 항목 | 내용 |
|------|------|
| **반품 신청 기간** | 수령일로부터 14일 이내에 마이페이지 또는 고객센터(1234‑5678)로 신청 가능 |
| **반품 배송비** | <u>단순 변심</u> 시 고객 부담 (왕복)  <br>하자·오배송 시 위니브마켓 전액 부담 |
| **반품이 불가한 상품** | 신선식품, 냉장·냉동 식품, 사용으로 가치가 현저히 감소한 경우 |
| **교환** | 색상/사이즈 변경 한 번 무료 제공<br>다른 상품 교환은 반품 후 재주문으로 처리 |
| **환불 시점** | 물류센터에 도착해 검수 완료 후, 영업일 기준 3일 이내에 결제 수단으로 환불 |

위 내용이 반품·교환에 관한 모든 안내입니다.


## 🔟 LLM 단독 vs RAG 비교 ⚖️

같은 질문을 **문서 없이 그냥 LLM에게** 물어본 답과, **RAG** 답을 비교해 봅시다.
문서를 건네주는 것만으로 답이 어떻게 달라지는지 확인하세요.

In [36]:
question = "위니브마켓 반품 정책이 뭐야?"

# ① LLM 단독 (문서 없이)
alone = llm.invoke(question).content

# ② RAG (문서 검색 후)
rag = rag_chain.invoke(question)

print("❌ [LLM 단독]  — 문서 없이, 자기 지식으로")
print(alone)
print("\n" + "=" * 55 + "\n")
print("✅ [RAG]  — 반품 정책 문서를 근거로")
print(rag)

❌ [LLM 단독]  — 문서 없이, 자기 지식으로
### 위니브마켓(위니브마켓·Wini B Market) 반품·교환 정책 (2026 년 기준)

> **※※** ※※  
> 정책은 상품 종류(식품·화장품·잡화 등)와 판매 채널(온라인·오프라인·공식몰·서드파티 플랫폼)마다 조금씩 다를 수 있으니, **구매하신 곳**에서 제공하는 리턴 안내 페이지를 꼭 참고하세요. 아래 내용은 대표적인 사례를 요약한 것입니다.

| 항목 | 상세 내용 |
|------|-----------|
| **1. 반품·교환 기간** | - **온라인(개인몰, 공통몰)**: 구매일로부터 **7 일 이내**.<br>- **오프라인 매장**: 상품 수령일로부터 **7 일 이내** (매장 내에서만 가능).<br>- **특정 할인·세일 상품**: 일반 반품 기간 보다 **30 일** 이내에 반품이 가능. |
| **2. 반품 자격** | - **제품 손상·위생**: 포장 라벨, 비닐, 원래 상자 등이 모두 원본인 경우.<br>- **기본 가치 보존**: 사용/열람 흔적이 없는 상태, 부착 라벨이 그대로 있는 경우.<br>- **사이즈·색상 실수**: 구매 후 7 일 이내에 반품/교환 가능. |
| **3. 반품 절차** | 1. **구매 내역 확인** → “주문/배송 조회” 페이지에서 반품 신청.<br>2. **라벨 부착** → 추가 라벨이 필요한 경우, “반품 배송 라벨”을 인쇄 후 묶을 것.<br>3. **포장** → 원래 패키지(혹은 포장지)를 그대로 사용.<br>4. **반품 택배** → “쇼핑몰 택배 콜리스택”에 부착 후 지정 장소에 반품. (택배비가 발생할 수 있음; 결제 시 ‘환불 택배비 무상’ 옵션이 있는 경우 무료 반송) |
| **4. 환불·교환** | - **해외 상품**: 환불은 결제 통화·수단으로 그대로 처리.<br>- **환불**: 가까운 금융기관 계좌명·계좌번호 입력, 기간은 반품 접수 후 **3–7 영업일** 내.<br>- **교환**: 반품 통보 시 원하는 제품으로

---
## 🎉 정리

우리는 5단계로 **문서 기반 챗봇**을 완성했습니다.

| 단계 | 한 일 |
| --- | --- |
| 1️⃣ 2️⃣ 3️⃣ | 문서를 청크로 분할 |
| 4️⃣ | 임베딩 → 벡터 DB 저장 |
| 5️⃣ | 리트리버로 검색 |
| 6️⃣ 7️⃣ | 프롬프트에 청크 삽입(증강) |
| 8️⃣ | LLM 준비 |
| 9️⃣ | LCEL로 연결 & 생성 |
| 🔟 | LLM 단독 vs RAG 비교 |

**핵심**: 모델의 기억이 아니라 **우리 문서**로 답하게 만드는 것이 RAG! 🚀

## ✏️ 과제

아래를 직접 바꿔가며 실험해 보세요. (다음 셀에서!)

1. **다른 질문 던지기** — `rag_chain.invoke("...")` 에 신선식품 반품, 환불 시점, 교환 등 다른 질문을 넣어보세요.
2. **문서에 없는 질문** — "적립금은 어떻게 쌓이나요?" 처럼 문서에 없는 내용을 물으면 어떻게 답하는지 확인해 보세요. (정말 "모른다"고 하나요?)
3. **`k` 바꾸기** — `vectorstore.as_retriever(search_kwargs={"k": 1})` 로 줄이면 답이 어떻게 달라지나요?
4. **`chunk_size` 바꾸기** — 2️⃣ 셀에서 `chunk_size=100` 으로 줄이면 청크가 몇 개로 나뉘나요? 검색 품질은 어떤가요?
5. **(심화) 내 문서로 바꾸기** — 1️⃣ 셀의 `policy_text` 를 여러분의 문서(회사 규정·위키 등)로 바꿔, 나만의 챗봇을 만들어 보세요.

In [37]:
# ✏️ 여기서 자유롭게 실험해 보세요!
# 예시)
print(rag_chain.invoke("신선식품도 반품되나요?"))

신선식품은 반품이 불가합니다. 해당 내용은 안내되어 있지 않습니다.


In [38]:
print(rag_chain.invoke("환불은 언제 되나요?"))

반품 상품이 물류센터에 도착해 검수가 완료되면, 영업일 기준 3일 이내에 결제하신 수단으로 환불이 진행됩니다.


In [39]:
print(rag_chain.invoke("적립금은 어떻게 쌓이나요?"))  # 문서에 없는 질문

해당 내용은 안내되어 있지 않습니다.
